## CS310 Natural Language Processing
## Lab 9: Reward Model for Human Alignment

In this lab, we will practice the following tasks:
- The code for training a reward model that assigns scores to pairs of sentences. 
- Loading and processing human preference data for training a reward model.
- Play with a trained reward model.

Install `peft`, `datasets` before getting started.
```bash
pip install peft datasets
```


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F


from transformers import Qwen3ForCausalLM, AutoTokenizer, AutoModelForCausalLM

## T1. Defining a Reward Model


We will use the [LlamaForCausalLM](https://huggingface.co/docs/transformers/model_doc/llama#transformers.LlamaForCausalLM) model from HuggingFace, as the basis for our reward model.

First, two internal forward functions are to be implemented:
- `_forward_rm`: it takes the input ids and attention masks of a sequence (user input + response), and returns the reward scores.
  - The reward scores are in tensor of same shape as the input ids, with **one reward score for each token**.
  - Reward scores are calculated by calling a linear layer `self.reward_head` on the last hidden state (of the entire sequence).
- `_forward_lmloss`: it takes the input of same format, but returns the regular language modeling loss.
  - Logits are computed by calling `self.lm_head` on the last hidden state.
  - The `response_ids` are used as the target for the `nn.CrossEntropyLoss()`.

Then, define the `forward` function, which takes the input ids and attention masks of two sequences, and returns the combined loss.
- Compute `reward1` on the first sequence (positve example) and `reward2` on the second sequence (negative example).
- Calculate their difference as `logits`
- Reward loss is computed by calling `F.binary_cross_entropy_with_logits(logits, label)`.
- Here, the `label` is the ground truth labels to indicate which direction of logits = $r^{+}-r^{-}$ should go. 1 to push the logits to positive values (this is what we want), and 0 to push the logits to be negative.

In [18]:
class Qwen3RewardModel(Qwen3ForCausalLM):
    def __init__(self, config):
        super().__init__(config)

        # A linear layer to map hidden states to a scalar, as the final reward
        self.reward_head = nn.Linear(config.hidden_size, 1, bias=False)

    def _forward_rm(self, input_ids, attention_mask, **kargs):
        """
        input_ids: input token ids
        attention_mask: attention mask
        Return: reward scores, output from self.reward_head
        """
        # Call original self.model.forward()  to get the hidden states
        output = self.model.forward(
            input_ids=input_ids,
            attention_mask=attention_mask, 
            return_dict=True,
            use_cache=False
        )
        ### START YOUR CODE ###
        # Feed the last hidden state from output to self.reward_head to get the reward score
        # output.last_hidden_state: (batch, seq_len, hidden_size)
        # reward_head maps to (batch, seq_len, 1) -> squeeze to (batch, seq_len)
        rewards = self.reward_head(output.last_hidden_state).squeeze(-1)
        ### END YOUR CODE ###

        return rewards 
    
    def _forward_lmloss(self, prompt_ids, lm_attn_mask, response_ids):
        """
        prompt_ids: input token ids
        lm_attn_mask: attention mask
        response_ids: target ids
        Return: cross-entropy loss for language modeling
        """ 
        # Call original self.model.forward()  to get the hidden states
        outputs = self.model.forward(
            input_ids=prompt_ids,
            attention_mask=lm_attn_mask,
            return_dict=True,
            use_cache=False,
        )
        criterion = nn.CrossEntropyLoss()

        ### START YOUR CODE ###
        # Get logits from lm_head: (batch, seq_len, vocab_size)
        logits = self.lm_head(outputs.last_hidden_state)
        # response_ids: (batch, resp_len) -> align with last resp_len tokens
        resp_len = response_ids.shape[-1]
        # Take the last resp_len token logits as predictions for the response
        logits_for_resp = logits[:, -resp_len:, :].contiguous()
        # Flatten for CrossEntropyLoss (N, C) vs (N,)
        loss = criterion(logits_for_resp.view(-1, logits_for_resp.size(-1)), response_ids.view(-1))
        ### END YOUR CODE ###

        return loss
        
    def forward(self, sent1_idx, attention_mask_1, \
        sent2_idx, attention_mask_2, 
        labels, prompt_ids, lm_attn_mask, response_ids, **kargs):
        """
        sent1_idx: User input ids + positive output ids
        attention_mask_1: Attention mask for sent1_idx
        sent2_idx: User input ids + negative output ids
        attention_mask_2: Attention mask for sent2_idx

        labels: The ground truth labels to indicate which direction of logits = reward_positive - reward_negative should go. 1 to push the logits to be positive, and 0 to push the logits to be negative.
        prompt_ids: User input ids + positive output ids
        lm_attn_mask: Attention mask for prompt_ids
        response_ids: Target ids for calculating cross-entropy loss
        """

        ### START YOUR CODE ###
        # Reward for positive example: per-token rewards (batch, seq_len)
        reward1 = self._forward_rm(sent1_idx, attention_mask_1)

        # Reward for negative example
        reward2 = self._forward_rm(sent2_idx, attention_mask_2)

        # Reduce per-token rewards to a single scalar per sample by masking and summing
        mask1 = attention_mask_1.float()
        mask2 = attention_mask_2.float()
        r1 = (reward1 * mask1).sum(dim=1)
        r2 = (reward2 * mask2).sum(dim=1)

        # Calculate the reward difference logits (batch,)
        logits = r1 - r2

        # Ensure labels are shaped (batch,) and float for BCEWithLogits
        lab = labels
        if lab.dim() > 1:
            lab = lab.view(lab.size(0), -1)[:, 0]
        lab = lab.float()

        # Compute the reward modeling loss
        rm_loss = F.binary_cross_entropy_with_logits(logits, lab)

        # Compute the language modeling loss 
        lm_loss = self._forward_lmloss(prompt_ids, lm_attn_mask, response_ids)
        ### END YOUR CODE ###

        # Final loss
        loss = rm_loss + lm_loss

        return loss

In [19]:
import os
torch.manual_seed(123)

MODEL_PATH = os.path.abspath('E:/CS310-Natural-Language-Processing/lab/lab7/Qwen3-0.6B')
reward_model = Qwen3RewardModel.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 2437.10it/s]
Qwen3RewardModel LOAD REPORT from: E:\CS310-Natural-Language-Processing\lab\lab7\Qwen3-0.6B
Key                | Status  | 
-------------------+---------+-
reward_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
# Test
torch.manual_seed(123)

sent1 = "This is a test sentence."
sent1_encoded = tokenizer(sent1, return_tensors='pt')

with torch.no_grad():
    reward1 = reward_model._forward_rm(**sent1_encoded)
print(reward1)

# You are expected to see the following output:
# tensor([[ 0.3262,  1.5312, -0.1543,  2.5938,  0.1855,  0.6289]],
#        dtype=torch.bfloat16)

tensor([[ 0.3281,  1.5078, -0.1157,  2.5312,  0.1445,  0.6055]],
       dtype=torch.bfloat16)


In [21]:
# 保证本单元可独立运行：若未定义 tokenizer 则自动加载（健壮性检测）
try:
    tokenizer
except NameError:
    from transformers import AutoTokenizer
    import os
    MODEL_PATH = os.path.abspath('E:/CS310-Natural-Language-Processing/lab/lab7/Qwen3-0.6B')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)

Right now the output reward scores don't have meanings as the model is not trained yet.

## T2. Preparing Data for Reward Model

We will load the preference dataset from `Anthropic/hh-rlhf` for some testing.

The following function prepares the preference dataset in a user-friendly view.

In [22]:
from datasets import load_dataset

DATA_PATH = './hh-rlhf'

def get_data(split, data_path):
    dataset = load_dataset(split=split, path=data_path)

    def split_prompt_and_responses_hh(sample):
        search_term = "\n\nAssistant:"
        search_term_idx = sample["chosen"].rfind(search_term)
        assert search_term_idx != -1, f"Prompt and response does not contain '{search_term}'"
        prompt = sample["chosen"][:search_term_idx + len(search_term)]
        return {
            "prompt": prompt,
            "chosen": sample["chosen"][len(prompt):],
            "rejected": sample["rejected"][len(prompt):],
        }

    return dataset.map(split_prompt_and_responses_hh)

In [23]:
# Print some data
train_dataset = get_data('train', DATA_PATH)

prompt = train_dataset[0]['prompt']
chosen = train_dataset[0]['chosen']
rejected = train_dataset[0]['rejected']

print("Prompt:", prompt[:45])
print("Chosen:", chosen)
print("Rejected:", rejected)

Prompt: 

Human: What are some cuss words in english?
Chosen:  I haven't even thought about it.
Rejected:  Ass.


Now, let's prepare a single data instance to be fed to the reward model for training.

- `sent1_encoded` is the tokenized result of `prompt + chosen` (positive example)
- `sent2_encoded` is the tokenized result of `prompt + rejected` (negative example)

The two tokenized sequences (input ids and attention masks) should be padded to same length.

In [24]:
### START YOUR CODE ###
# Run tokenizer
sent1_encoded = tokenizer(prompt + chosen, return_tensors='pt')
sent2_encoded = tokenizer(prompt + rejected, return_tensors='pt')

# Pad input ids to same length
sent1_idx = sent1_encoded['input_ids']
sent2_idx = sent2_encoded['input_ids']
max_len = max(sent1_idx.size(-1), sent2_idx.size(-1))
pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
if sent1_idx.size(-1) < max_len:
    sent1_idx = F.pad(sent1_idx, (0, max_len - sent1_idx.size(-1)), value=pad_token_id)
if sent2_idx.size(-1) < max_len:
    sent2_idx = F.pad(sent2_idx, (0, max_len - sent2_idx.size(-1)), value=pad_token_id)

# Pad attention masks
sent1_attn_mask = sent1_encoded['attention_mask']
sent2_attn_mask = sent2_encoded['attention_mask']
if sent1_attn_mask.size(-1) < max_len:
    sent1_attn_mask = F.pad(sent1_attn_mask, (0, max_len - sent1_attn_mask.size(-1)), value=0)
if sent2_attn_mask.size(-1) < max_len:
    sent2_attn_mask = F.pad(sent2_attn_mask, (0, max_len - sent2_attn_mask.size(-1)), value=0)
### END YOUR CODE ###

# Test
print(sent1_idx.shape)
print(sent2_idx.shape)
print(sent1_attn_mask.shape)
print(sent2_attn_mask.shape)

# You are expected to see the following output:
# torch.Size([1, 185])
# torch.Size([1, 185])
# torch.Size([1, 185])
# torch.Size([1, 185])

torch.Size([1, 185])
torch.Size([1, 185])
torch.Size([1, 185])
torch.Size([1, 185])


Put the above token ids and masks into a dictionary as a data instance.

- `sent1_idx` and `sent2_idx` are the token ids of the positive and negative examples, respectively.
- `attention_mask_1` and `attention_mask_2` are the attention masks of the positive and negative examples, respectively.
- `labels` is a tensor of all ones with the same shape as `sent1_idx`.
- `prompt_ids`, `lm_attn_mask`, and `response_ids` are the token ids, attention mask, and target ids of the positive example, respectively.



In [25]:
input_data = {
    ### START YOUR CODE ###
    'sent1_idx': sent1_idx, 
    'attention_mask_1': sent1_attn_mask, 
    'sent2_idx': sent2_idx, 
    'attention_mask_2': sent2_attn_mask, 
    'labels': torch.ones(sent1_idx.shape, dtype=torch.float), 

    'prompt_ids': sent1_idx, 
    'lm_attn_mask': sent1_attn_mask, 
    'response_ids': tokenizer(chosen, return_tensors='pt')['input_ids'],
    ### END YOUR CODE ###
}

In [26]:
# Test
torch.manual_seed(123)

with torch.no_grad():
    output = reward_model(**input_data)
    print(output)

# You are expected to see a single loss value
# tensor(14.6250, dtype=torch.bfloat16)

# Runtime Error is likely to because by the implementation of the internal forward functions
# You can examine if the internal reward model forward function is implemented correctly
# r1 = model._forward_rmloss(sent1_idx, sent1_attn_mask)
# print(r1.shape)

tensor(13.2500)


## T3. Play with a Trained Reward Model

Next, we will load a trained reward model [j1-nano](https://github.com/haizelabs/j1-micro/tree/master) and examine its output on some sample data. 

**j1-nano** is a reward model trained on top of Qwen3-0.6B, using LoRA as the parameter effective fine-tuning (PEFT). 

From `j1-nano-06B/adapter_config.json` you can see the key configurations: 
`"peft_type": "LORA",
 "r": 16,`, ...

The model can be loaded using the `peft` library together with the loaded Qwen3-0.6B model.

In [27]:
# First, load the Qwen3-0.6B as the base model
import os
MODEL_PATH = os.path.abspath('E:/CS310-Natural-Language-Processing/lab/lab7/Qwen3-0.6B')
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
base_model_path = MODEL_PATH # "Qwen/Qwen3-0.6B", or specify your own path

base_model = AutoModelForCausalLM.from_pretrained(base_model_path,
    torch_dtype=torch.bfloat16, # don't change this
    device_map="auto", # don't change this
    local_files_only=True,
    trust_remote_code=True,
    )

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4841.67it/s]


In [28]:
# Next, load j1-nano using peft
from peft import PeftModel

adapter_path = './j1-nano-0.6B'

j1_model = PeftModel.from_pretrained(base_model, adapter_path, local_files_only=True)
j1_tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True, trust_remote_code=True) # also, the tokenizer is a bit different from the original one due to PEFT.

Next, `peft` provides an option to merge the LoRA weights to the origianl model by calling `.merge_and_unload()`.

It would internally update the original model weights by $W = W_{base} + B\cdot A$, and result in slightly faster inference. Here, `B` and `A` are the LoRA matrices.


In [29]:
j1_model = j1_model.merge_and_unload()
j1_model.eval() # it is important to set the model to evaluation mode, so the dropout is switched off.

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

You can use the reward model just as if a regular language model. 

Try generating 100 tokens without using random sampling.

In [30]:
sentence = "The weather is sunny today."

input_encoded = j1_tokenizer(sentence, return_tensors='pt').to(j1_model.device)

output = j1_model.generate(
    input_ids=input_encoded.input_ids,
    attention_mask=input_encoded.attention_mask,
    max_new_tokens=100,
    do_sample=False,
)

print(j1_tokenizer.decode(output[0], skip_special_tokens=False))

The weather is sunny today. The temperature is 20 degrees Celsius. The sun is shining, and the wind is blowing. What is the total number of degrees Celsius that the sun has been shining for today? To find the total number of degrees Celsius that the sun has been shining for today, we need to consider the temperature and the time it has been shining. However, since the temperature is given as 20 degrees Celsius, and the sun is shining for a certain duration, we can assume that the total number of


To unleash the **j1-nano**'s true power, we will feed in some a query and two candidate responses as input for the model to judge. 

**j1-nano** is not trained to directly output the reward score as it is done in our `T1`example, using an additional `reward_head`. 

Instead, it can follow the judging instruction, and directly output formatted reward scores for both candidate responses. This is because the base model (Qwen3-0.6B) already has very strong instruction-following capabilities. 

In [31]:
query = "Explain the theory of relativity in simple terms."

response_a = (
    "Relativity is about how time and space change depending on how fast you're moving. "
    "Einstein showed that the faster you go, the slower time passes for you compared to someone standing still."
)
response_b = "It's a physics thing by Einstein."

Providing the model with some prompts:

In [32]:
import json
from utils import judge_system_prompt, judge_prompt_format

messages = [
    {"role": "system", "content": judge_system_prompt()},
    {"role": "user", "content": judge_prompt_format(query, response_a, response_b)},
]

print(json.dumps(messages, indent=2))
print(messages)

[
  {
    "role": "system",
    "content": "You are an expert XML wrangler. You must respond in the following format, regardless of the input:\n\n<specific_criteria>\n...\n</specific_criteria>\n<analysis>\n...\n</analysis>\n<scores>\n\\boxed{{..., ...}}\n</scores>\n\nPlease only respond in English."
  },
  {
    "role": "user",
    "content": "You are a skilled little expert at scoring responses. You should evaluate given responses based on the given judging criteria.\nGiven the context of the conversation (the last round is the User's query) and multiple responses from the Assistant, you need to refer to the [General Evaluation Criteria] to score the responses. Based on the general evaluation criteria, state potential other specific criteria to the query, the weights of different criteria, and then provide an overall comprehensive score upon them.\nEach score is an integer between 1 and 10, with a higher score indicating that the response meets the relevant criteria more closely. For 

You can see that sophisticated rubrics are used to guide the model to carry out the judging task.

Next, fill in the following code to generate the output of the judging task. 

This time, generate `1024` tokens, because it is running under *thinking* mode, and will need cosume a lot more tokens.

In [33]:
text = j1_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = j1_tokenizer(text, return_tensors="pt").to(j1_model.device)

In [34]:
with torch.no_grad():
    outputs = j1_model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1024,
        do_sample=False,
    )
# It will take ~1 minute to generate on a CPU.

Let's look at the raw output:

In [35]:
raw_response = j1_tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print("=" * 60)
print("RAW MODEL OUTPUT:")
print("=" * 60)
print(raw_response)
print("=" * 60)

RAW MODEL OUTPUT:
<think>
Okay, let's see. I need to evaluate the responses to the question about explaining the theory of relativity in simple terms. The user wants to know how to approach this.

First, let's look at the response A. It says relativity is about time and space changing based on movement, and Einstein showed that moving faster means time passes slower. That's a good start, but maybe I should check if it's accurate. I remember that time dilation is a key concept here, but maybe this response is too brief and misses some details. Let's see, if I were to rate this, maybe it's partially adhered because it's missing some specifics.

Then, Response B is just a statement that it's a physics thing by Einstein. That doesn't explain relativity, so I think that's not adhered. It might be considered not adhered because it's too vague.

Now, applying the evaluation criteria. For Instruction Adherence, Response B doesn't meet because it's vague. Response A might be partially adhered i

We can see that the the final reward scores are generated between the `<scores></scores>` tags, and in a `\boxed` format.

This judging output can be further extracted using some regular expression-based search:

In [36]:
from utils import extract_scores

try:
    scores = extract_scores(raw_response)
    print(f"Score for Response A: {scores[0]}")
    print(f"Score for Response B: {scores[1]}")
except Exception as e:
    print(f"Failed to extract scores: {e}")

Score for Response A: 6.0
Score for Response B: 1.0


The scores make sense, because Response A is indeed a more informative one than Response B.